In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager
from openpyxl import Workbook
import time

#options 설정
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

#Service 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

#크롤링 설정
url = "https://www.musinsa.com/products/5075748"
driver.get(url)

#어디에서 오류가 나는지 확인하기 위해 try문 실행
# 스냅후기라는 button 찾아오기
try :
    review_tab = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), '스냅·후기')]"))
        #특정 요소가 존재하고, 보이고, 클릭할 수 있는 상태인지 확용
        # 웹사이트를 열고, 스냅/후기 버튼을 찾고 클릭할 수 있을 때까지 기다린다
        # 버튼이 준비되면 클릭되지만 10초 내에 클릭되지 않으면 실패라고 뜨게 하는 것이다
        #클래스는 바뀌어도 태그값은 바뀌지 않기 때문에 xpath 사용
        # 튜플 형태의 인자값 밖에 못 받음, 클릭할 수 있는 요소가 여러 개가 나올 수 있기 때문
    )
    review_tab.click()
    time.sleep(2)
except Exception as e :
    print("리뷰 탭 클릭 실패 :", e)

scroll_pause = 1
last_height = driver.execute_script("return document.body.scrollHeight")
for _ in range(5) :
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    time.sleep(scroll_pause)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height :
        break
    last_height = new_height

soup = BeautifulSoup(driver.page_source, "html.parser")

# product_name = soup.select_one("span.text-title_18px_med.sc-1omefes-1.exqQRL.font-pretendard").get_text(strip = true)
product_name = soup.select_one("span.text-title_18px_med.sc-1omefes-1.exqQRL.font-pretendard").get_text().strip()

review_cards = soup.select("div.GoodsReviewStaticList__Container-sc-s1ucwv-0.lnqPxe")
reviews = []

for card in review_cards :
    text_tags = card.select("span.text-body_13px_reg.text-black.font-pretendard")
    for tag in text_tags : 
        text = tag.get_text(strip = True)
        if text and len(text) > 10 :
            reviews.append(text)

reviews = list(set(reviews)) 
#set : 중복을 허용하지 않는 집합으로 변환해서 같은 텍스트가 여러 번 들어가는 걸 제거
#list : 다시 리스트로 감싸서 기존처럼 인덱싱이나 반복이 가능하게끔 함

print(f"상품명 :  {product_name}")
for idx, review in enumerate(reviews, start = 1) :
    print(f"{idx}, {review}")

상품명 :  키라키라 자수 반팔티 화이트
1, 자수인점이 좋았고 귀여워요 ㅋㅋ 회사에 입고가긴 부끄럽지만 주말에 아주 잘 입고 있습니다
2, 고양이 자수가 귀여워요 한여름에 입기엔 살짝 더울 것 같아요
3, 생각보다 엄청 루즈핏이라 크네요
그래도 뒤에 자수가 너무 귀여워서 맘에들어요
4, 심플하면서 자수 그래픽이라 특이해서 자주 입게 됩니다
5, 항상 티 퀄리티는 믿고구매합니다. 자수도 딴딴해서 좋고 세탁할때도 걱정이 안됩니다. 오버핏으로 넉넉하게 좋습니다
6, 화이트 티에 포인트로 들어간 키라키라 자수가 진짜 귀엽고 감성 있어요. 과하지 않은 크기인데도 존재감 확실해서 심심한 코디에 은근한 포인트 주기 좋습니다. 가까이서 보면 자수 디테일이 섬세해서 퀄리티도 만족스러
7, 면이 두깨가있어요. 귀여워요. 빨면 주름져서 건조기말고 그냥말리는게나아요
8, 요즘 티셔츠 브랜드 중에서 제일 좋아하는 브랜드가 됐어요. 지난 한 달 동안 4개 정도 샀어요 ㅋㅋ 품질과 소재도 좋고, 핏도 마음에 들어요!
9, 여름마다 반팔 필요할때 찾는 브랜드 bb 편하고 이쁨
10, 좋은 천연 소재라 피부에 닿는 느낌이 좋아요. 디자인이 프린트가 아니라 원단에 꿰맨 것에 덕분에 훨씬 보기 좋아요! 품질이 정말 좋아서 이 가게에서 티셔츠를 여러 장 샀어요.
